## EEG recording and KAFFEE generation

Run this notebook to render an EEG reconstruction example (KAFFEE vs. ground truth)

Panel (a) is a 4s snippet of the standardized recording used for training. Panel (b) is a freely generated shell-on rollout after a 0.25T = 2410-step burn-in phase (same that is used to compute the stochastic long-run/dynamical fidelity scores).

The figure is written to `figures/eeg/eeg_recording_vs_generated.pdf`.


In [1]:
from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

%matplotlib inline

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from data.eeg import SAMPLING_RATE_HZ, load_eeg_recording
from models.shplrnn_shell import ShallowPLRNNShell, stochastic_rollout

_spec = importlib.util.spec_from_file_location(
    "paper_figure_style", REPO_ROOT / "figures" / "paper_style.py"
)
_paper_style = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_paper_style)


In [ ]:
SUBJECT = "S001"
METHOD = "kaffee"
SNIPPET_SECONDS = 8.0
BURN_STEPS = 2410
ROLLOUT_SEED_BASE = 50_000
TRACE_OFFSET = 5.0
TRACE_LW = 1.0
RECORDING_COLOR = "#000000"
KAFFEE_COLOR = "#009E73"
FONT_WIDTH = 6.58
CANVAS_WIDTH = 6.85
INCHES_PER_FOUR = 2.18


# Channel labels from S001R01 of the PhysioNet EEG MMI dataset (https://physionet.org/content/eegmmidb/1.0.0/#description,doi:10.13026/C28G6P). Each entry is (channel name, column in dataset) listed from the front of the head to the back.

CHANNELS = (
    ("Fp1", 21), ("Fpz", 22), ("Fp2", 23),
    ("Af7", 24), ("Af3", 25), ("Afz", 26), ("Af4", 27), ("Af8", 28),
    ("F7", 29), ("F5", 30), ("F3", 31), ("F1", 32), ("Fz", 33),
    ("F2", 34), ("F4", 35), ("F6", 36), ("F8", 37),
    ("Ft7", 38), ("Ft8", 39),
    ("Fc5", 0), ("Fc3", 1), ("Fc1", 2), ("Fcz", 3), ("Fc2", 4), ("Fc4", 5), ("Fc6", 6),
    ("T9", 42), ("T7", 40), ("C5", 7), ("C3", 8), ("C1", 9), ("Cz", 10),
    ("C2", 11), ("C4", 12), ("C6", 13), ("T8", 41), ("T10", 43),
    ("Cp5", 14), ("Cp3", 15), ("Cp1", 16), ("Cpz", 17), ("Cp2", 18), ("Cp4", 19), ("Cp6", 20),
    ("Tp7", 44), ("Tp8", 45),
    ("P7", 46), ("P5", 47), ("P3", 48), ("P1", 49), ("Pz", 50),
    ("P2", 51), ("P4", 52), ("P6", 53), ("P8", 54),
    ("Po7", 55), ("Po3", 56), ("Poz", 57), ("Po4", 58), ("Po8", 59),
    ("O1", 60), ("Oz", 61), ("O2", 62), ("Iz", 63),
)

# Choose eight channels for the appx. figure.
PAPER_CHANNELS = ("Fpz", "Fz", "Cz", "T7", "T8", "Pz", "O1", "Oz")
FIGURE_DIR = REPO_ROOT / "figures" / "eeg"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def channels_by_name(names: tuple[str, ...]) -> tuple[int, ...]:
    index = {name: column for name, column in CHANNELS}
    missing = [name for name in names if name not in index]
    if missing:
        raise KeyError(f"Unknown series: {missing}")
    return tuple(index[name] for name in names)


def channel_names(n_channels: int) -> tuple[str, ...]:
    """Evenly spaced names in plot order. """
    names = tuple(name for name, _ in CHANNELS)
    if len(names) != 64:
        raise RuntimeError(f"Expected 64 series, found {len(names)}")
    if n_channels == 8:
        return PAPER_CHANNELS
    if n_channels >= 64:
        return names
    picked = np.unique(np.rint(np.linspace(0, 63, n_channels)).astype(int))
    if picked.size != n_channels:
        raise ValueError(f"Selection collapsed to {picked.size} for n={n_channels}")
    return tuple(names[int(i)] for i in picked)


def median_seed() -> int:
    """We choose the KAFFEE seed whose D_stsp is closest to the across-seed median."""
    medians = []
    for seed in range(20):
        path = (
            REPO_ROOT / "models" / "eeg_main_shells" / SUBJECT
            / f"seed{seed:02d}" / METHOD / "eval.json"
        )
        payload = json.loads(path.read_text())
        medians.append((seed, float(np.median(payload["D_stsp_stoch_repeats"]))))
    center = float(np.median([value for _, value in medians]))
    seed, value = min(medians, key=lambda item: (abs(item[1] - center), item[0]))
    print(
        f"S001 KAFFEE seed {seed:02d}: median D_stsp={value:.3f}, "
        f"cross-seed median {center:.3f}"
    )
    return seed


def stacked(ax, series, times, channels, color, labels) -> None:
    n_channels = len(channels)
    for row, channel in enumerate(channels):
        ax.plot(
            times,
            series[:, channel] + (n_channels - 1 - row) * TRACE_OFFSET,
            color=color,
            lw=TRACE_LW,
        )
    ax.set_yticks([(n_channels - 1 - row) * TRACE_OFFSET for row in range(n_channels)])
    ax.set_yticklabels(labels)
    ax.set_xlim(float(times[0]), float(times[-1]))
    ax.set_ylabel("channel")


def pad_shared_ylim(axes) -> None:
    lows, highs = [], []
    for ax in axes:
        for line in ax.get_lines():
            y = np.asarray(line.get_ydata(), dtype=np.float64)
            lows.append(float(np.min(y)))
            highs.append(float(np.max(y)))
    lo, hi = min(lows), max(highs)
    axes[0].set_ylim(lo - 0.45 * TRACE_OFFSET, hi + 0.55 * TRACE_OFFSET)


def plot_eeg_comparison(recorded, generated, times, names, stem: str) -> Path:
    channels = channels_by_name(names)
    canvas = (CANVAS_WIDTH, INCHES_PER_FOUR * len(names) / 4)
    points = _paper_style.points(FONT_WIDTH)
    with plt.rc_context(_paper_style.rc_params(FONT_WIDTH)):
        fig, axes = plt.subplots(1, 2, figsize=canvas, sharey=True)
        stacked(axes[0], recorded, times, channels, RECORDING_COLOR, names)
        stacked(axes[1], generated, times, channels, KAFFEE_COLOR, names)
        pad_shared_ylim(axes)
        axes[1].set_ylabel("")
        axes[0].set_title("Recording", pad=12)
        axes[1].set_title("KAFFEE", pad=12)
        for ax, letter in zip(axes, ("(a)", "(b)")):
            ax.text(
                -0.02, 1.10, letter, transform=ax.transAxes,
                ha="right", va="bottom", fontsize=points["panel"], fontweight="bold", clip_on=False,
            )
        fig.tight_layout()
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        tick_bottom = min(
            lab.get_window_extent(renderer).y0
            for ax in axes
            for lab in ax.get_xticklabels()
            if lab.get_text()
        )
        tick_bottom_fig = fig.transFigure.inverted().transform((0.0, tick_bottom))[1]
        left = axes[0].get_position()
        right = axes[1].get_position()
        fig.text(
            0.5 * (left.x0 + right.x1),
            tick_bottom_fig - 10.0 / (fig.get_figheight() * 72.0),
            "time (s)",
            ha="center",
            va="top",
            fontsize=points["label"],
        )
        path = FIGURE_DIR / stem
        fig.savefig(path.with_suffix(".pdf"), bbox_inches="tight", pad_inches=0.01)
        fig.savefig(path.with_suffix(".png"), dpi=200, bbox_inches="tight", pad_inches=0.01)
        plt.close(fig)
    print(f"wrote {path.with_suffix('.pdf').relative_to(REPO_ROOT)}  {', '.join(names)}")
    return path.with_suffix(".pdf")


In [ ]:
torch.set_num_threads(2)
seed = median_seed()
recording = load_eeg_recording(SUBJECT)
shell = ShallowPLRNNShell.load(
    REPO_ROOT / "models" / "eeg_main_shells" / SUBJECT / f"seed{seed:02d}" / METHOD / "shell_bundle.pt",
    method_key=METHOD,
    device="cpu",
)
n_show = int(SNIPPET_SECONDS * SAMPLING_RATE_HZ)
generated = stochastic_rollout(
    shell,
    recording[0],
    burn_steps=BURN_STEPS,
    score_steps=n_show,
    seed=ROLLOUT_SEED_BASE + seed,
)
recorded = recording[BURN_STEPS : BURN_STEPS + n_show]
times = np.arange(n_show) / SAMPLING_RATE_HZ
print(f"snippet {n_show} samples, {SNIPPET_SECONDS:.0f} s at {SAMPLING_RATE_HZ:.0f} Hz")


In [ ]:
paper_pdf = plot_eeg_comparison(
    recorded, generated, times, PAPER_CHANNELS, "eeg_recording_vs_generated"
)


In [ ]:
for n_channels in (16, 32, 64):
    plot_eeg_comparison(
        recorded,
        generated,
        times,
        channel_names(n_channels),
        f"eeg_recording_vs_generated_ch{n_channels}",
    )
